# YOLO11 NPU + ByteTrack — 객체 추적(tracking) 데모

ARIES NPU가 **검출**(YOLO11), CPU가 **추적**(경량 ByteTrack, `yolo_npu/track.py`). 영상 → track ID.

- 검출만 NPU, 추적(칼만+IoU+헝가리안)은 CPU로 충분 → 실시간(샘플 25fps@1카드).
- 커널 `pe_npu_host` (qbruntime + scipy + opencv). 공개 샘플 영상 사용.
- ByteTrack: 고신뢰 검출로 1차 매칭 → 남은 트랙을 저신뢰 검출로 2차(가림에 강함).

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.getcwd())))  # 레포 루트
import cv2, matplotlib.pyplot as plt
from yolo_npu import YOLONPU, ByteTrack, draw_tracks

VIDEO = '../../scratch_yolo/people-detection.mp4'        # 공개 샘플
def show(img, title=None):
    plt.figure(figsize=(11,6)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis('off');
    if title: plt.title(title)
    plt.show()

## 1. 영상 추적 — 프레임마다 검출→추적, 출력 영상 저장

`det(frame)` → `trk.update(boxes)` → `draw_tracks`. `--classes 0`(person)만 추적.

In [ ]:
det = YOLONPU.load("yolo11m", "single", conf_thres=0.25)   # HF 먼저→없으면 컴파일 안내
trk = ByteTrack(fps=12); ByteTrack.reset_ids()
cap = cv2.VideoCapture(VIDEO)
W, H = int(cap.get(3)), int(cap.get(4))
writer = cv2.VideoWriter('track_out.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 12, (W, H))

best, seen = None, set()
while True:
    ok, frame = cap.read()
    if not ok: break
    boxes = [b for b in det(frame) if b[5] == 0]     # person
    tracks = trk.update(boxes)
    for *_ , tid, _, _ in tracks: seen.add(tid)
    vis = draw_tracks(frame, tracks, det.names)
    writer.write(vis)
    if len(tracks) >= 3 and best is None: best = vis.copy()   # 시각화용 한 장
cap.release(); writer.release()
print(f'누적 track ID {len(seen)}개 → track_out.mp4')
if best is not None: show(best, '사람별 track ID (#id)')

## 참고
- 실시간·멀티스트림이면 `YOLONPU(device_ids='auto')` + 스트림별 `ByteTrack` 인스턴스.
- 파라미터: `ByteTrack(track_thresh=, match_thresh=, track_buffer=, fps=)` — 가림 많으면 `track_buffer`↑.
- 외형(ReID) 기반이 필요하면 ReID CNN도 NPU로 컴파일해 붙일 수 있음(YOLO와 동일 흐름).